# 04 — Segmentação de Produtos (KMeans)

Este notebook executa segmentação via KMeans usando um pipeline (Imputer → StandardScaler → KMeans), calcula Elbow + Silhouette e exporta o modelo e a base com clusters.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
MODELS_DIR = REPORTS_DIR / 'models'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.clustering import fit_kmeans_pipeline, name_clusters, save_pipeline, score_k_range


## Carregar base com PSI

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'base_produtos_psi.csv')
df.shape


## Elbow + Silhouette (k=2..10)

In [ ]:
features = ['discounted_price_clean','discount_pct_clean','rating_clean','rating_count_clean','PSI']
metrics = score_k_range(df, features=features, k_values=range(2, 11))
metrics


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(metrics['k'], metrics['inertia'], marker='o')
plt.title('Elbow Curve (Inertia)')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kmeans_elbow.png', dpi=160)
plt.close()

plt.figure(figsize=(10,4))
plt.plot(metrics['k'], metrics['silhouette'], marker='o')
plt.title('Silhouette Score by k')
plt.xlabel('k')
plt.ylabel('Silhouette')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kmeans_silhouette.png', dpi=160)
plt.close()


## Treinar modelo final e exportar artefatos

In [ ]:
best_k = int(metrics.sort_values('silhouette', ascending=False).iloc[0]['k'])
best_k


In [ ]:
artifacts = fit_kmeans_pipeline(df, features=features, n_clusters=best_k)
df_out = df.copy()
df_out['cluster'] = artifacts.labels.values
df_out = pd.concat([df_out, artifacts.pca_2d], axis=1)
cluster_names = name_clusters(df_out, cluster_col='cluster')
df_out['cluster_name'] = df_out['cluster'].map(cluster_names)
df_out[['cluster','cluster_name']].value_counts().reset_index(name='n').head(10)


In [ ]:
save_pipeline(artifacts.pipeline, str(MODELS_DIR / 'kmeans_pipeline.joblib'))
df_out.to_csv(PROCESSED_DIR / 'base_produtos_com_clusters.csv', index=False)
df_out.to_csv(REPORTS_DIR / 'base_produtos_com_clusters.csv', index=False)
(MODELS_DIR / 'kmeans_pipeline.joblib', PROCESSED_DIR / 'base_produtos_com_clusters.csv')
